# IHC-Global analysis: data collection and aggregation

First step in IHC analysis. The workflow is as follows:

1 - Run this script, which combines all recordingsd in three files, master , alltraces, allCorrTraces, which contains all cells and all traces/correlationTraces (correlationTraces are timeseries representing the local correlation of the pixels of one ROI over time. Note if you rerun this code, by default it copy over the cells already analysed from the old file (produced by traceExplorer, see below) and just adds the new ones.

2 - Run the IHC-Deconvolution-Cascade notebook to predict the spike probability using cascade for all the traces. 

3 - Use the traceExplorer GUI (in src) to open, the master , alltraces, allCorrTraces and label the peaks semiautomatically.

4 - Run the IHC-GlobalAnalysis-part2 - dataProcessing for further analysis (e.g., selection of high quality peaks etc..)

In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
%pylab
%load_ext autoreload
%autoreload 2
%matplotlib inline
import pandas as pd
import os
from scipy.signal import savgol_filter,find_peaks
from scipy.signal import argrelextrema,argrelmin,argrelmax
import sys
sys.path.append('../../src')
import seaborn as sns

import ast 
from movieTools import thorlabsFile
from tifffile import imread

parameterFolder = '../../parameters/'

fileHeader = [
'alpha9'
]


#Load and concatenate all the files with the IHC experiments. Create unique mouse IDs by appending the strain to the mouse number
alldata = []
for h in fileHeader:
    inputFilename = os.path.join('../../',h)+'.xlsx'

    alldata1 = pd.read_excel(inputFilename)
    alldata.append(alldata1[alldata1['discard']!=1])


alldata = pd.concat(alldata, ignore_index=True)
alldata = alldata[~alldata['Folder'].isna()]
alldata = alldata.reset_index()

alldata = alldata.sort_values(['Age','Mouse ID','Folder']).reset_index()


#Folders to load and save the data from
resultsFOlder = '../../analysis_results/'
fileHeader = 'alpha9'

inputFilename = os.path.join(resultsFOlder,fileHeader)+'.xlsx'

tempFilename = '../../analysis_results/master_temp.csv'

Using matplotlib backend: module://matplotlib_inline.backend_inline
%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


# Some functions first

In [3]:
from skimage import measure
from movieTools import determineCellTypes, determineCentroids
from scipy.interpolate import interp1d
from traceUtilities import fillMissingValues

def extractTraces(folder,fps = 1,returndFF0 = True,returnCorrelations = True, fillMissingValue = False):
    filename = os.path.join(folder,'processedMovies','traces.csv')
    traces = pd.read_csv(filename,index_col=0)
    x = traces['Frame']/fps
    traces = traces.drop('Frame',axis=1)
    if fillMissingValue:
        traces = fillMissingValues(traces)
    stop = int(traces.shape[0]/2)
    f0 = traces.iloc[:stop,:].quantile(0.3)
    
    dff0 = (traces - f0)/f0



    if returnCorrelations:
        filename = os.path.join(folder,'processedMovies','traces_localCorr.csv')
        correlations = pd.read_csv(filename,index_col=0)
        if returndFF0:
            return x,dff0,correlations
        else:
            return x,traces,correlations

    else:
        if returndFF0:
            return x,dff0
        else:
            return x,traces

def extractImages(folder):
    filename = os.path.join(folder,'processedMovies','Masks.tif')
    filename2 =  os.path.join(folder,'processedMovies','Annotations.tif')
    filename3 =  os.path.join(folder,'processedMovies','Avg.tif')
    masks = imread(filename)
    annotations = imread(filename2)
    avg = imread(filename3)

    return (avg,masks,annotations)

def imshowImage(avg,masks,annotations, highlighCell = None):
    if highlighCell is None:
        masks2 = masks.copy().astype(float)
        masks2[masks==0] = np.nan
        figure()
        imshow(avg,cmap=cm.gray)
        masks2 = annotations.copy().astype(float)
        masks2[masks==0] = np.nan
        imshow(masks2)
    else:
        masks2 = masks.copy().astype(float)
        masks2[masks==0] = np.nan
        figure()
        imshow(avg,cmap=cm.gray)
        masks2 = annotations.copy().astype(float)
        masks2[masks!=highlighCell] = np.nan
        imshow(masks2)

def renderAnnotations(tb,avg,masks,annotations):
    try:
        l = tb.app.layers['Avg']
        l.data = avg
    except:
        tb.app.add_image(avg,name='Avg',visible=False)
    try:
        l = tb.app.layers['Masks']
        l.data = masks
    except:
        tb.app.add_labels(masks,name='Masks',visible=False)
    
    try:
        l = tb.app.layers['Annotations']
        l.data = annotations
    except:
        tb.app.add_labels(annotations,name='Annotations',visible=False)

from scipy.signal import periodogram
def noise_amplitude(trace):
    y = periodogram(trace,window='flattop',scaling='spectrum')[1]
    halfL = int(len(y)/2)
    return np.sqrt(y[halfL:].mean())*np.sqrt(2)

# Create a database of all the traces and of all the identified events, or load from a file

In [4]:
#Change drive 
localDrive = '/media/marcotti-lab' #D:
alldata['Folder'] = localDrive + alldata['Folder'].str[2:].str.replace('\\','/')

In [5]:
from tqdm.auto import tqdm
def createMaster(alldata, inputFilename=inputFilename, overrideSave=False):
    '''
    This function will go through the recordings and create a master where every row is a ROI. 
    Initially it will load a previous file (inputfilename) if it exist, and add ROIs to it only if they don't exist already. 
    If overrideSave is true it will save the files even if no new cells have been added, otherwise it will save the file only
    if new cells are present
    '''
    try:
        oldmaster = pd.read_excel(inputFilename)
        oldmaster['Peak positions'] = oldmaster['Peak positions'].apply(ast.literal_eval)
    except FileNotFoundError:
        print('No result file found')
        oldmaster = pd.DataFrame(columns=['Cell ID'])
    new_cells = 0
    master = pd.DataFrame(columns=['Cell ID'])
    alltraces = pd.DataFrame()
    allCorrTraces = pd.DataFrame()

    pbar = tqdm(
        alldata.iterrows(),
        total=len(alldata),
        desc='Processing recordings',
        unit='rec',
        dynamic_ncols=True
    )
    for i,el in pbar:
        current_folder = str(el['Folder']).split('/')[-1]
        pbar.set_postfix_str(current_folder)

        try: 
            x,t = extractTraces(el['Folder'],el['fps'], fillMissingValue= True, returnCorrelations=False)
            correlations = None
            avg,masks,annotations = extractImages(el['Folder'])
            celltypes = determineCellTypes(masks,annotations)
            centroids = determineCentroids(masks) 
            if t.shape[1]!= celltypes.size:
                print(t.shape)
                print(celltypes.size)
                print(el['Folder']+' Error: mismatch number of cells')
                break
                
            for j in range(t.shape[1]):
                cellID = str(el['Mouse ID'])+'_'+el['Folder'].split('\\')[-1] + '_' + t.columns[j]
                
                alltraces = pd.concat((alltraces,pd.DataFrame({cellID:t.iloc[:,j]})),axis=1)
                if correlations is not None:
                    allCorrTraces = pd.concat((allCorrTraces,pd.DataFrame({cellID:correlations.iloc[:,j]})),axis=1)
                if cellID not in oldmaster['Cell ID'].values: # If the cell already exists in the db, don't load it.
                    this_master = pd.DataFrame(columns = [ 'Peak positions'])
                    this_master['Peak positions'] = this_master['Peak positions'].astype(object) 

                    this_master.loc[0,'Folder'] = el['Folder']
                    this_master.loc[0,'RoiN'] = t.columns[j]
                    this_master.loc[0,'Cell type'] = celltypes[t.columns[j]]
                    this_master.loc[0,'Mouse ID'] = el['Mouse ID']
                    this_master.loc[0,'Age'] = el['Age']
                    this_master.loc[0,'fps'] = el['fps']
                    # this_master.loc[0,'agent'] = el['agent']
                    # this_master.loc[0,'oxygen delivery'] = el['oxygen delivery']
                    this_master.loc[0,'Peak prominence'] = 6
                    this_master.loc[0,'Peak min distance'] = 1
                    this_master.loc[0,'Peak min height'] = 0
                    this_master.loc[0,'Peak correlation'] = 0
                    this_master.loc[0,'Use MAD z-score'] = True
                    this_master.loc[0,'Min duration (s)'] = 0.2

#                    this_master.at[0,'Peak positions'] = list(find_peaks(t.iloc[:,j],prominence=0.2)[0])

                    this_master.loc[0,'Centroid x'] = centroids[t.columns[j]][0]
                    this_master.loc[0,'Centroid y'] = centroids[t.columns[j]][1]
                    this_master.loc[0,'Centroid x um'] = centroids[t.columns[j]][0] * el['um/pixel']
                    this_master.loc[0,'Centroid y um'] = centroids[t.columns[j]][1] * el['um/pixel']

                    this_master.loc[0,'Independent recordings number'] = el['Independent recordings number']
                    this_master.loc[0,'Number in sequence'] = el['Number in sequence']

                    this_master.loc[0,'Cell ID'] = cellID
                    new_cells = new_cells + 1

                else:
                    this_master = pd.DataFrame(columns = [ 'Peak positions'])
                    this_master['Peak positions'] = this_master['Peak positions'].astype(object) 

                    this_master.loc[0,'Folder'] = el['Folder']
                    this_master.loc[0,'RoiN'] = t.columns[j]
                    this_master.loc[0,'Cell type'] = celltypes[t.columns[j]]
                    this_master.loc[0,'Mouse ID'] = el['Mouse ID']
                    this_master.loc[0,'Age'] = el['Age']
                    this_master.loc[0,'fps'] = el['fps']
                   # this_master.loc[0,'agent'] = el['agent']
                   # this_master.loc[0,'oxygen delivery'] = el['oxygen delivery']
                    if oldmaster.loc[oldmaster['Cell ID']==cellID, 'Peak prominence'].shape[0]!=1:
                        print('Problem')
                    this_master.loc[0,'Peak prominence'] =oldmaster.loc[oldmaster['Cell ID']==cellID, 'Peak prominence'].values[0]
                    this_master.loc[0,'Peak min distance'] = oldmaster.loc[oldmaster['Cell ID']==cellID, 'Peak min distance'].values[0]
                    this_master.loc[0,'Peak min height'] = oldmaster.loc[oldmaster['Cell ID']==cellID, 'Peak min height'].values[0]
                    this_master.loc[0,'Peak correlation'] = oldmaster.loc[oldmaster['Cell ID']==cellID, 'Peak correlation'].values[0] 
                    this_master.loc[0,'Use MAD z-score'] = oldmaster.loc[oldmaster['Cell ID']==cellID, 'Use MAD z-score'].values[0] 
                    this_master.loc[0,'Min duration (s)'] = oldmaster.loc[oldmaster['Cell ID']==cellID, 'Min duration (s)'].values[0] 


                    this_master.at[0,'Peak positions'] = oldmaster.loc[oldmaster['Cell ID']==cellID, 'Peak positions'].values[0] #list(find_peaks(t.iloc[:,j],prominence=0.2)[0])

                    this_master.loc[0,'Centroid x'] = centroids[t.columns[j]][0]
                    this_master.loc[0,'Centroid y'] = centroids[t.columns[j]][1]
                    this_master.loc[0,'Centroid x um'] = centroids[t.columns[j]][0] * el['um/pixel']
                    this_master.loc[0,'Centroid y um'] = centroids[t.columns[j]][1] * el['um/pixel']

                    this_master.loc[0,'Independent recordings number'] = el['Independent recordings number']
                    this_master.loc[0,'Number in sequence'] = el['Number in sequence']

                    this_master.loc[0,'Cell ID'] = cellID

                master = pd.concat((master,this_master),axis=0,ignore_index=True)

        except FileNotFoundError:
            print('File not found: '+el['Folder'])
            
    print('Added '+str(new_cells)+' cells.')
    master['Discard'] = np.nan
    #Save the file again if new cells are found
    if new_cells!=0 or overrideSave:
        master.to_excel(inputFilename)
        alltraces.to_csv(os.path.join(resultsFOlder,'alpha9_events_alltraces.csv'))
        allCorrTraces.to_csv(os.path.join(resultsFOlder,'alpha9_events_allCorrTraces.csv'))
    return master, alltraces, allCorrTraces


# Load directly the results from a previously saved file or recompute the master,traces and corrTraces from scratch (uncomment as required)

In [ ]:
#Recreate from scratch. If a inputfileName is provided, previously identified calcium events will be carried over 
master , alltraces, allCorrTraces = createMaster(alldata,overrideSave=False)

No result file found


Processing recordings:   0%|          | 0/435 [00:00<?, ?rec/s]

File not found: /media/marcotti-lab/Users/Rinta/Data/2025/In vivo/2025122_P7_In vivo_C57_6N_AAV_syn_setup 1/1
File not found: /media/marcotti-lab/Users/Rinta/Data/2025/In vivo/2025122_P7_In vivo_C57_6N_AAV_syn_setup 1/3
File not found: /media/marcotti-lab/Users/Francesca/Data/2025/2025_09_08_Myo15-GCaMP6f_P8_in vivo/10_002
File not found: /media/marcotti-lab/Users/Rinta/Data/2025/In vivo/2025123_P8_In vivo_C57_6N_AAV_syn_setup 1/1
File not found: /media/marcotti-lab/Users/Rinta/Data/2025/In vivo/2025123_P8_In vivo_C57_6N_AAV_syn_setup 1/1_001
File not found: /media/marcotti-lab/Users/Rinta/Data/2025/In vivo/2025123_P8_In vivo_C57_6N_AAV_syn_setup 1/1_002
File not found: /media/marcotti-lab/Users/Rinta/Data/2025/In vivo/2025123_P8_In vivo_C57_6N_AAV_syn_setup 1/2
File not found: /media/marcotti-lab/Users/Rinta/Data/2025/In vivo/2025123_P8_In vivo_C57_6N_AAV_syn_setup 1/3_001
File not found: /media/marcotti-lab/Users/Rinta/Data/2025/In vivo/2025124_P9_In vivo_AAV synGCAMP_c576N_setup1/1
